Imports

In [1]:
import torch
import torch.nn as nn
import math

Positional Encoding

In [10]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # [1, max_len, d_model]

    def forward(self, x):
        # x: [B, T, d_model]
        return x + self.pe[:, :x.size(1)]

Multi-head attention

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h = n_heads
        self.dk = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, q_in, kv_in, mask=None):
        B, Tq, _ = q_in.shape
        Tk = kv_in.shape[1]

        Q = self.q_proj(q_in).view(B, Tq, self.h, self.dk).transpose(1, 2)   # [B,h,Tq,dk]
        K = self.k_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)  # [B,h,Tk,dk]
        V = self.v_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)  # [B,h,Tk,dk]

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.dk)   # [B,h,Tq,Tk]

        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        attn = scores.softmax(dim=-1)
        out = attn @ V                                             # [B,h,Tq,dk]
        out = out.transpose(1, 2).contiguous().view(B, Tq, -1)      # [B,Tq,d_model]
        return self.out_proj(out)

Causal Mask

In [3]:
def causal_mask(T, device):
    # True = blocked position
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

In [5]:
m = causal_mask(5, 'cpu')
print(m)

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])


FFN

In [6]:
class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, d_model),
        )
    def forward(self, x):
        return self.net(x)

One decoder layer - self-attn, cross-attn, FFN, each with residual + norm

In [7]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, ff_dim)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, memory, self_mask):
        x = x + self.self_attn(self.norm1(x), self.norm1(x), mask=self_mask)
        x = x + self.cross_attn(self.norm2(x), memory, mask=None)
        x = x + self.ffn(self.norm3(x))
        return x

Full decoder — embedding + positional encoding + stacked layers + output head

In [8]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024, max_len=50):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, ff_dim) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, tgt_in, memory):
        B, T = tgt_in.shape
        x = self.embed(tgt_in) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        mask = causal_mask(T, tgt_in.device)

        for layer in self.layers:
            x = layer(x, memory, mask)

        x = self.norm_out(x)
        return self.fc_out(x)   # [B, T, vocab_size]

In [11]:
decoder = Decoder(vocab_size=39, d_model=256, n_heads=8, n_layers=3)

dummy_memory = torch.randn(4, 32, 256)     # pretend encoder output
dummy_tgt_in = torch.randint(0, 39, (4, 6))  # pretend token ids, batch of 4, length 6

logits = decoder(dummy_tgt_in, dummy_memory)
print(logits.shape)   # expect [4, 6, 39]

torch.Size([4, 6, 39])


In [12]:
decoder.eval()
with torch.no_grad():
    tgt = torch.randint(0, 39, (1, 5))
    mem = torch.randn(1, 32, 256)

    out_full = decoder(tgt, mem)

    tgt_truncated = tgt.clone()
    tgt_truncated[0, 3:] = 0   # zero out positions 3,4 — shouldn't matter for position 2's output
    out_trunc = decoder(tgt_truncated, mem)

    print(torch.allclose(out_full[0, 2], out_trunc[0, 2], atol=1e-5))   # should be True

True
